# Mejora 2: aumento de datos diseñado para pastillas

**Faster R-CNN v2 (ResNet-50 + FPN) sobre *pills-sxdht*.** Segundo experimento de mejora.

## Hipótesis

El baseline se entrena con **solo 315 imágenes y sin ningún aumento de datos**: en cada época el modelo ve exactamente las mismas imágenes. Con tan pocos datos, el riesgo es que memorice detalles de cada foto (posición, orientación, iluminación, fondo) en lugar de aprender qué es cada pastilla. Tras la mejora 1, el análisis de errores deja dos problemas a los que el aumento de datos puede ayudar:

1. **Localización** (`loc`): sigue siendo el error más costoso a IoU 0.85 (ΔAP +0.063 en test, +0.030 en validación). Si el modelo ve cada pastilla en distintas orientaciones, posiciones y tamaños, la regresión de cajas generaliza mejor que si memoriza la caja exacta de cada imagen.
2. **Falsos positivos sobre el fondo** (`bg`): en la imagen 81 de test, los lunares azul claro del fondo se detectan como pastillas `white` (5 detecciones con score ≥ 0.5 en la mejora 1, hasta 0.92). El modelo se ha quedado con "círculo claro" en vez de con la pastilla. Más variedad de encuadres, escalas e iluminación debería hacerle depender menos de esos atajos.

**Lo que el aumento de datos *no* puede arreglar: las 2 pastillas "no detectadas" de test** (`miss`, +0.011). Al revisarlas en la mejora 1 resultaron ser **errores de anotación** de la imagen 74: la pastilla rosa está anotada dos veces sobre la misma caja (`Ibuphil Cold 400-60`, correcta, y `Cipro 500`), y hay una caja `Cipro 500` de 0.5×2 píxeles. Ningún modelo puede acertarlas, así que `miss` no debería cambiar. Por lo mismo, el **mAP de objetos pequeños de test no es interpretable**: solo hay 2 objetos pequeños y uno es esa caja degenerada.

## Diseño del aumento: qué transformaciones y por qué

La clave del diseño es que **no todas las transformaciones son válidas en este dataset**: cuatro de las siete clases (`blue`, `pink`, `red`, `white`) se definen **por el color**. Un aumento "genérico" que cambie el tono convertiría una pastilla roja en rosa sin cambiar su etiqueta, y enseñaría al modelo algo falso.

| Transformación | Parámetros | Justificación |
|---|---|---|
| Volteo horizontal y vertical | p = 0.5 cada uno | Las fotos son primeros planos cenitales: una pastilla no tiene "arriba" ni "abajo". |
| Rotación de 0°, 90°, 180° o 270° | aleatoria | Igual que el volteo. Solo múltiplos de 90° porque así **la caja girada sigue siendo exacta**; con un ángulo arbitrario, la caja que envuelve la caja girada es más grande que la pastilla y se enseñaría al modelo a dibujar cajas holgadas, justo el error `loc` que queremos reducir. |
| Escala y traslación | escala 0.8–1.2, traslación ±10 % | Pastillas de más tamaños y en más posiciones, y fondos encuadrados de otra forma (problemas 1 y 2). Las cajas que quedan parcialmente fuera se recortan al borde, y las que quedan casi fuera se eliminan. |
| Brillo y contraste | ±20 % | Distinta iluminación al hacer la foto. Mantiene el color de la pastilla. |
| Saturación | ±10 % (suave) | Pequeñas variaciones de la cámara, sin llegar a desteñir una pastilla de color hasta parecer `white`. |
| ~~Tono (*hue*)~~ | **no se usa** | Cambiaría la clase de las pastillas de color. |
| ~~Rotación arbitraria, *mosaic*, recortes agresivos~~ | **no se usan** | Cajas inexactas o pastillas cortadas: perjudicarían la localización. |

El aumento **solo se aplica a train**. Validación y test se evalúan con las imágenes originales, igual que en el baseline.

## Qué cambia respecto al baseline

| Aspecto | Baseline | Mejora 2 |
|---|---|---|
| Aumento de datos | ninguno | **el de la tabla anterior** |
| Épocas y política de LR | 15, LR constante | **las de la mejora 1** (30 épocas, *warmup* + coseno) si la mejora 1 funcionó; si no, las del baseline. Se elige con `USE_MEJORA1_RECIPE`. |
| Todo lo demás | | igual |

Se parte de la receta de la mejora 1 porque el aumento de datos hace el problema de entrenamiento más difícil (cada época las imágenes son distintas) y **suele necesitar más épocas** para dar fruto. Por eso, el efecto del aumento se mide **frente a la mejora 1**, que solo difiere en el aumento. La comparación con el baseline mide el efecto acumulado.

## Cómo se comprueba

- **Métrica de decisión**: mAP@0.5:0.95 en **validación**.
- **Hipótesis**: a IoU 0.85 debería bajar el ΔAP de `loc` respecto a la mejora 1, y deberían bajar los falsos positivos de fondo con score alto (imagen 81, fila *fondo* de la matriz de confusión). `miss` no debería cambiar (son errores de anotación).
- **Sobreajuste**: la distancia entre la pérdida de train y la de validación debería ser menor que sin aumento.

## 1. Configuración

Los parámetros marcados con `# CAMBIO` son los que difieren del baseline. `USE_MEJORA1_RECIPE` decide la receta de entrenamiento: déjalo en `True` si la mejora 1 mejoró el mAP de validación del baseline y cámbialo a `False` si no fue así.

In [ ]:
# Dependencias que pueden no estar instaladas
try:
    import datasets, torchmetrics, pycocotools  # noqa: F401
except ImportError:
    %pip install -q datasets torchmetrics pycocotools

import sys
import urllib.request
from pathlib import Path

if not Path("common.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HectorBarro/CV1_P6/main/faster_rcnn/common.py", "common.py")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision.transforms import v2
from torchvision.utils import draw_bounding_boxes

import common as C

%matplotlib inline

In [ ]:
EXP_NAME = "mejora02_augmentation_fasterrcnn"

# ---------------- Receta de entrenamiento ----------------
USE_MEJORA1_RECIPE = True   # True: 30 épocas + warmup + coseno (mejora 1) | False: 15 épocas, LR constante (baseline)

SEED = 42
BATCH_SIZE = 4
LR = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
if USE_MEJORA1_RECIPE:
    NUM_EPOCHS, WARMUP_EPOCHS, WARMUP_FACTOR, MIN_LR_FACTOR = 30, 1, 0.001, 0.0
    REFERENCE = "mejora01_cosine_fasterrcnn"   # experimento que solo difiere en el aumento de datos
else:
    NUM_EPOCHS = 15
    REFERENCE = "baseline_fasterrcnn"

# ---------------- Aumento de datos (CAMBIO: el baseline no tiene) ----------------
AUG = dict(
    flip_p=0.5,                 # volteo horizontal y vertical, cada uno con esta probabilidad
    rot90=True,                 # rotación aleatoria de 0, 90, 180 o 270 grados
    scale=(0.8, 1.2),           # zoom aleatorio
    translate=(0.1, 0.1),       # desplazamiento máximo (fracción del ancho y alto)
    brightness=0.2,
    contrast=0.2,
    saturation=0.1,
    hue=0.0,                    # NO: el color define 4 de las 7 clases
    min_box_size=4,             # cajas recortadas por el borde más pequeñas que esto (px) se eliminan
)

DESCRIPTION = (f"Aumento de datos (volteos, rot90, escala {AUG['scale']}, traslación, brillo/contraste, "
               f"saturación suave, sin hue) | " + ("receta mejora 1" if USE_MEJORA1_RECIPE else "receta baseline"))

OUT_DIR = C.RESULTS_ROOT / EXP_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

C.set_seed(SEED)
print("Dispositivo:", C.describe_device())
print("Referencia para medir el efecto del aumento:", REFERENCE)

## 2. Transformaciones de entrenamiento

Se usan las transformaciones `v2` de torchvision, que transforman **la imagen y las cajas a la vez** (las cajas van como `tv_tensors.BoundingBoxes` en el `target`).

- `RandomRotation` con un ángulo fijo de 90°·k y `expand=True`: para múltiplos de 90° la caja rotada es exacta.
- Tras el `RandomAffine` (escala + traslación), `ClampBoundingBoxes` recorta las cajas al borde de la imagen y `SanitizeBoundingBoxes` elimina las degeneradas (y sus etiquetas).
- El color se modifica al final, sobre la imagen ya en `float` en [0, 1].

In [ ]:
def build_train_transforms(aug):
    tfs = [v2.ToImage()]
    if aug["flip_p"] > 0:
        tfs += [v2.RandomHorizontalFlip(aug["flip_p"]), v2.RandomVerticalFlip(aug["flip_p"])]
    if aug["rot90"]:
        tfs.append(v2.RandomChoice([v2.RandomRotation((a, a), expand=True) for a in (0, 90, 180, 270)]))
    if aug["scale"] != (1.0, 1.0) or aug["translate"] != (0.0, 0.0):
        tfs.append(v2.RandomAffine(degrees=0, translate=aug["translate"], scale=aug["scale"]))
    tfs += [
        v2.ClampBoundingBoxes(),
        v2.SanitizeBoundingBoxes(min_size=aug["min_box_size"]),  # también elimina las etiquetas de esas cajas
        v2.ToDtype(torch.float32, scale=True),
        v2.ColorJitter(brightness=aug["brightness"], contrast=aug["contrast"],
                       saturation=aug["saturation"], hue=aug["hue"]),
    ]
    return v2.Compose(tfs)


train_tf = build_train_transforms(AUG)
print(train_tf)

## 3. Datos y modelo

Mismas particiones, limpieza de clases y modelo que el baseline. La única diferencia es la transformación de train. Validación y test usan la transformación del baseline.

In [ ]:
hf, CLASSES = C.load_splits()
NUM_CLASSES = len(CLASSES)

model = C.build_model(NUM_CLASSES, seed=SEED)

ds = C.make_datasets(hf, train_tf=train_tf)
loaders = C.make_loaders(ds, BATCH_SIZE, SEED)
print(f"Imágenes -> train: {len(ds['train'])} | valid: {len(ds['valid'])} | test: {len(ds['test'])}")

### 3.1 Comprobación visual del aumento

Antes de entrenar hay que **verificar que las cajas siguen a las pastillas** después de cada transformación: un error aquí (por ejemplo, cajas que no rotan con la imagen) estropearía el entrenamiento sin dar ningún error. Se muestra la misma imagen de train 4 veces, cada una con un aumento aleatorio distinto, junto a la original.

In [ ]:
def show_augmentations(dataset, idx, n=5, seed=0):
    torch.manual_seed(seed)
    fig, axes = plt.subplots(1, n + 1, figsize=(3.4 * (n + 1), 3.6))
    tf_backup = dataset.transforms
    dataset.transforms = C.base_transforms()
    views = [("original", dataset[idx])]
    dataset.transforms = tf_backup
    views += [(f"aumento {k + 1}", dataset[idx]) for k in range(n)]
    for ax, (title, (img, tgt)) in zip(axes, views):
        canvas = draw_bounding_boxes((img * 255).to(torch.uint8), tgt["boxes"], colors="lime", width=3,
                                     labels=[CLASSES[l] for l in tgt["labels"].tolist()], font_size=12)
        ax.imshow(canvas.permute(1, 2, 0))
        ax.set_title(f"{title} | {tuple(img.shape[1:])}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    return fig


for i in [0, 7, 21]:
    fig = show_augmentations(ds["train"], i, n=4, seed=i)
    fig.savefig(OUT_DIR / f"ejemplos_aumento_{i}.png", dpi=100)
    plt.show()

## 4. Optimizador, política de LR y entrenamiento

Igual que la receta elegida en la sección 1. Mismo criterio de selección: el modelo con **mayor mAP@0.5:0.95 en validación**.

> El aumento se hace en CPU con `num_workers=0`, así que cada época puede tardar algo más que en el baseline (~160 s en una T4).

In [ ]:
C.set_seed(SEED)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

if USE_MEJORA1_RECIPE:
    iters_per_epoch = len(loaders["train"])
    scheduler = C.warmup_cosine_scheduler(optimizer, NUM_EPOCHS * iters_per_epoch, WARMUP_EPOCHS * iters_per_epoch,
                                          warmup_factor=WARMUP_FACTOR, min_factor=MIN_LR_FACTOR)
else:
    scheduler = None

fit_result = C.fit(model, loaders, optimizer, NUM_EPOCHS, OUT_DIR, scheduler=scheduler)
history, best_epoch = fit_result["history"], fit_result["best_epoch"]

## 5. Curvas de entrenamiento

Se comparan con el baseline y con la referencia (la mejora 1, si se usa su receta). Qué mirar:

- **Pérdida de train más alta** que sin aumento: es lo esperado, porque las imágenes cambian en cada época y el problema es más difícil.
- **Distancia entre train y validación**: si el aumento reduce el sobreajuste, la pérdida de validación debería quedar más cerca de la de train.
- **mAP de validación**: puede subir más despacio al principio y superar a la referencia al final.

Si los resultados de la mejora 1 no están en local ni en el repositorio, solo se compara con el baseline.

In [ ]:
refs = {"baseline": C.load_experiment("baseline_fasterrcnn")}
if REFERENCE != "baseline_fasterrcnn":
    refs["mejora 1"] = C.load_experiment(REFERENCE)
refs = {k: v for k, v in refs.items() if v["results"] is not None}

b = refs["baseline"]
b["history"].setdefault("lr", [b["results"]["config"]["lr"]] * len(b["history"]["val_map"]))

C.plot_history_comparison({**{k: v["history"] for k, v in refs.items() if v["history"]}, "mejora 2 (aumento)": history},
                          out_path=OUT_DIR / "curvas_vs_referencias.png")

for name, h in [*((k, v["history"]) for k, v in refs.items() if v["history"]), ("mejora 2", history)]:
    gap = h["val_loss"][-1] - h["train_loss"][-1]
    print(f"{name:<10} | mejor val mAP {max(h['val_map']):.4f} | loss train {h['train_loss'][-1]:.4f} | "
          f"loss val {h['val_loss'][-1]:.4f} | val - train {gap:+.4f}")

## 6. Evaluación en test

In [ ]:
res, per_class = C.test_report(model, loaders["test"], CLASSES)
C.plot_pr_curves(res, CLASSES, out_path=OUT_DIR / "curvas_pr_test.png")

times_ms = C.measure_inference(model, ds["test"])
print(f"\nHardware: {C.hardware_name()} | inferencia: {times_ms.mean():.1f} ± {times_ms.std():.1f} ms/imagen")

### 6.1 Comparación

Entre paréntesis, la diferencia con la **primera columna (baseline)**. Para aislar el efecto del aumento hay que comparar la columna de la mejora 2 con la de la mejora 1. Con 45 imágenes de validación y 89 de test, diferencias de menos de 1 punto de mAP pueden deberse al azar.

El AP de objetos pequeños no se usa para sacar conclusiones: en test solo hay 2 objetos pequeños y uno es una caja mal anotada de 0.5×2 píxeles.

In [ ]:
config = {"seed": SEED, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE, "lr": LR,
          "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY, "optimizer": "SGD",
          "scheduler": ("warmup lineal 1 época + cosine, por iteración" if USE_MEJORA1_RECIPE else None),
          "augmentation": AUG}
summary = C.make_summary(EXP_NAME, DESCRIPTION, config, fit_result, res, per_class, times_ms)
C.compare_metrics({**{k: v["results"] for k, v in refs.items()}, "mejora 2": summary})

## 7. Análisis de errores (TIDE)

Se repite el análisis de la sección 13 del baseline y se guardan las mismas figuras (`ap_vs_iou_<split>.png`, `tide.png`, `matriz_confusion.png`, `peores_imagenes_test.png`), para poder ponerlas junto a las del baseline y la mejora 1:

- **TIDE**: según la hipótesis, a IoU 0.85 debería bajar `loc`. `miss` no debería cambiar (errores de anotación de la imagen 74).
- **AP frente a IoU**: se superpone la curva de la referencia (si sus predicciones están disponibles). Si mejora la localización, la curva de la mejora 2 cae más tarde.
- **Matriz de confusión** (score ≥ 0.5): la fila *fondo* cuenta los falsos positivos con score alto, como los de la imagen 81. También muestra si el ajuste de color afecta a las confusiones entre pastillas blancas.
- **Imágenes con más errores**: para comprobar visualmente si desaparecen los falsos positivos sobre los lunares.

In [ ]:
split_data = {
    "valid": C.collect_predictions(model, loaders["valid"], CLASSES),
    "test": C.collect_predictions(model, loaders["test"], CLASSES),
}
C.save_predictions(split_data, OUT_DIR)

tide = C.tide_analysis(split_data)
print()
C.compare_tide({**{k: v["tide"] for k, v in refs.items()}, "mejora 2": C.tide_to_json(tide)},
               keys=("test@0.5", "valid@0.85", "test@0.85"))

In [ ]:
ref_preds = {}
if REFERENCE != "baseline_fasterrcnn":
    ref_preds["mejora 1"] = C.load_predictions(REFERENCE)   # None si no están disponibles

C.error_analysis_figures(split_data, tide, CLASSES, ds["test"], OUT_DIR, label="mejora 2",
                         references=ref_preds)

## 8. Guardar resultados

In [ ]:
C.save_experiment(OUT_DIR, summary, tide)

if Path("/kaggle/working").exists():
    import shutil
    from IPython.display import FileLink

    shutil.make_archive(f"/kaggle/working/{EXP_NAME}", "zip", OUT_DIR)
    display(FileLink(f"{EXP_NAME}.zip"))

## 9. Conclusiones

> **Completar tras ejecutar el cuaderno**, con los números de las secciones 5, 6.1 y 7:
>
> - ¿Sube el mAP@0.5:0.95 de **validación** frente a la referencia (mejora 1 o baseline)?
> - ¿Baja `loc` a IoU 0.85 respecto a la mejora 1? ¿Cambian `bg` y `cls`? (`miss` debería seguir en 2: errores de anotación)
> - ¿Se reduce la distancia entre la pérdida de train y la de validación (menos sobreajuste)?
> - ¿Desaparecen los falsos positivos sobre el fondo de lunares de la imagen 81? (fila *fondo* de la matriz de confusión)
>
> **Si no mejora**, posibles causas a discutir: el aumento es demasiado fuerte para 30 épocas, la escala genera pastillas cortadas por el borde, o el dataset ya es muy homogéneo entre train y test y el aumento no aporta información nueva. Recordar también que con 45 imágenes de validación el mAP oscila ±1–2 puntos entre épocas (ver mejora 1). Se puede repetir con un aumento más suave (por ejemplo, solo volteos y rotaciones: `scale=(1.0, 1.0)`, `translate=(0.0, 0.0)`) para saber qué parte ayuda y cuál perjudica.